# AI-Powered Coding Tutor — Dual-Model Embedding Pipeline (Dynamic & Production Ready)

Pipeline Embedding tự động hoàn toàn (Dynamic Pipeline) triển khai trên Kaggle Cloud GPU (Tesla T4):
- **Text Embedder (Qwen3):** `Qwen/Qwen3-Embedding-0.6B` (Dim: 1.024, Context: 32k, Last-Token Pooling, Left-Padding, L2 Normalization).
- **Code Embedder (Jina Code):** `jinaai/jina-embeddings-v2-base-code` (Dim: 768, Context: 8k, Attention-Masked Mean Pooling, L2 Normalization).
- **Bộ định tuyến tự động:** `ModalityRouter(code_ratio_threshold=0.50)` phân chia động TEXT và CODE.
- **Cơ chế Checkpoint Resume thông minh:** Tự động phát hiện và bỏ qua các chunks đã hoàn thành, chỉ chạy các chunks mới bổ sung.
- **Tính toán động (Dynamic Scaling):** Tự động thích ứng với mọi kích thước dataset (`all_chunks.json` 5.230 chunks hoặc mở rộng thêm trong tương lai).


In [ ]:
# =============================================================================
# 1. KIỂM TRA PHẦN CỨNG GPU & CUDA
# =============================================================================
import os
import sys
import torch
import gc

print("=== BÁO CÁO PHẦN CỨNG & MÔI TRƯỜNG KAGGLE ===")
print(f"  • Python Version : {sys.version.split()[0]}")
print(f"  • PyTorch Version: {torch.__version__}")
print(f"  • CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    total_mem = torch.cuda.get_device_properties(current_device).total_memory / (1024 ** 3)
    print(f"  • Số lượng GPU   : {device_count}")
    print(f"  • GPU Hiện tại   : [{current_device}] {device_name}")
    print(f"  • VRAM Tổng      : {total_mem:.2f} GB")
    assert "T4" in device_name or "P100" in device_name or "V100" in device_name or "A100" in device_name or total_mem >= 14.0, \
        f"Cảnh báo: GPU {device_name} có thể không đủ VRAM tối ưu!"
else:
    raise SystemError("❌ LỖI: Kaggle Accelerator chưa được bật GPU! Hãy vào menu 'Settings -> Accelerator' và chọn GPU T4 x 2.")

print("🎉 Môi trường GPU đã sẵn sàng cho Dual-Model Embedding!")


In [ ]:
# =============================================================================
# 2. CÀI ĐẶT DEPENDENCIES TƯƠNG THÍCH CHO QWEN3 & JINA CODE
# =============================================================================
!pip install "transformers>=4.51.0" "sentence-transformers" "accelerate" "einops"

import transformers
import sentence_transformers
import accelerate
from packaging import version

print("\n✅ ĐÃ HOÀN TẤT CÀI ĐẶT VÀ XÁC THỰC PHIÊN BẢN:")
print(f"  • transformers         : {transformers.__version__} (Hỗ trợ kiến trúc Qwen3)")
print(f"  • sentence-transformers: {sentence_transformers.__version__}")
print(f"  • accelerate           : {accelerate.__version__}")

if version.parse(transformers.__version__) < version.parse("4.51.0"):
    print(f"\n⚠️ CẢNH BÁO: transformers ({transformers.__version__}) < 4.51.0 do kernel cache bản cũ!")
    print("👉 Hãy chọn 'Kernel -> Restart' trên thanh menu Kaggle rồi chạy tiếp để nạp bản mới!")
else:
    print("🎉 Xác nhận: Thư viện tương thích hoàn toàn với cả Qwen3 và Jina Code!")


In [ ]:
# =============================================================================
# 3. THIẾT LẬP ĐƯỜNG DẪN INPUT & OUTPUT (TỰ ĐỘNG DÒ TÌM FILE CHUNKS)
# =============================================================================
import os
from pathlib import Path

INPUT_CHUNK_FILE = None

# Danh sách các đường dẫn ưu tiên
priority_paths = [
    Path("/kaggle/input/datasets/minhthuy70/embedding/all_chunks.json"),
    Path("/kaggle/input/embedding/all_chunks.json"),
    Path("/kaggle/input/ai-coding-tutor-chunks/all_chunks.json"),
    Path("/kaggle/input/all_chunks.json"),
    Path("data/06_rag_dataset/chunks/all_chunks.json"),
    Path("/kaggle/input/datasets/minhthuy70/embedding/pilot_chunks_160.json"),
    Path("/kaggle/input/embedding/pilot_chunks_160.json"),
    Path("data/06_rag_dataset/chunks/pilot_chunks_160.json")
]

for pp in priority_paths:
    if pp.exists() and pp.is_file():
        INPUT_CHUNK_FILE = pp
        break

# Nếu chưa thấy, tự động quét đệ quy thư mục /kaggle/input
if INPUT_CHUNK_FILE is None and Path("/kaggle/input").exists():
    all_candidates = list(Path("/kaggle/input").rglob("all_chunks.json"))
    if not all_candidates:
        all_candidates = list(Path("/kaggle/input").rglob("*chunk*.json"))
    if all_candidates:
        INPUT_CHUNK_FILE = all_candidates[0]

if INPUT_CHUNK_FILE is None:
    print("❌ Không tìm thấy file chunk JSON nào trong /kaggle/input! Cây thư mục hiện tại:")
    if Path("/kaggle/input").exists():
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                print(f"  • {os.path.join(root, f)}")
    else:
        print("  (/kaggle/input không tồn tại trên máy local)")
    raise FileNotFoundError("Vui lòng add Dataset chứa file 'all_chunks.json' vào notebook qua nút 'Add Input' bên phải!")

print(f"📂 Đã xác định thành công file đầu vào: {INPUT_CHUNK_FILE} ({INPUT_CHUNK_FILE.stat().st_size:,} bytes)")

# Thiết lập các thư mục làm việc tại /kaggle/working/
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("data/08_rag_dataset")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
EMBEDDINGS_DIR = WORKING_DIR / "embeddings"
REPORTS_DIR = WORKING_DIR / "reports"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Thư mục checkpoints: {CHECKPOINT_DIR}")
print(f"📁 Thư mục embeddings : {EMBEDDINGS_DIR}")
print(f"📁 Thư mục reports    : {REPORTS_DIR}")


In [ ]:
# =============================================================================
# 4. CÁC LỚP TIỆN ÍCH CỐT LÕI (ROUTER, CHECKPOINT & VALIDATOR)
# =============================================================================
import json
import math
import time
import datetime
import re
from enum import Enum
from typing import List, Dict, Any, Tuple, Set

class Modality(str, Enum):
    TEXT = "text"
    CODE = "code"

class ModalityRouter:
    """Phân loại chunk thành TEXT hoặc CODE chuẩn hóa theo src/embedding/router.py:
    1. content_type == 'code_example' -> CODE (864 chunks)
    2. has_code == True AND code_char_ratio >= 0.50 -> CODE (73 chunks)
    3. Còn lại -> TEXT (4.293 chunks)
    Tổng: 937 CODE + 4.293 TEXT = 5.230 chunks.
    """
    def __init__(self, code_ratio_threshold: float = 0.50):
        self.code_ratio_threshold = code_ratio_threshold
        self._code_pattern = re.compile(r"```[a-zA-Z0-9_\-\.\+]*\s*([\s\S]*?)```", re.MULTILINE)

    def calculate_code_ratio(self, text: str) -> float:
        if not text:
            return 0.0
        matches = self._code_pattern.findall(text)
        if not matches:
            return 0.0
        return sum(len(m) for m in matches) / max(len(text), 1)

    def classify_chunk(self, chunk: Dict[str, Any]) -> Modality:
        metadata = chunk.get("metadata", {})
        content_type = metadata.get("content_type", chunk.get("content_type", ""))
        has_code = metadata.get("has_code", chunk.get("has_code", False))
        text = chunk.get("text", chunk.get("content", ""))

        # Quy tắc 1: Gắn nhãn sư phạm code_example (864 chunks)
        if content_type == "code_example":
            return Modality.CODE

        # Quy tắc 2: Chứa code fence và tỷ lệ ký tự code >= 50% (73 chunks)
        if has_code:
            ratio = self.calculate_code_ratio(text)
            if ratio >= self.code_ratio_threshold:
                return Modality.CODE

        return Modality.TEXT

    def route(self, chunks: List[Dict[str, Any]]) -> Dict[Modality, List[Dict[str, Any]]]:
        routed = {Modality.TEXT: [], Modality.CODE: []}
        for c in chunks:
            mod = self.classify_chunk(c)
            routed[mod].append(c)
        return routed

class VectorValidator:
    """Thẩm định chất lượng vector: dimension, hữu hạn (0 NaN/Inf), chuẩn hóa L2."""
    def __init__(self, expected_dim: int, l2_tolerance: float = 5e-3):
        self.expected_dim = expected_dim
        self.l2_tolerance = l2_tolerance

    def validate(self, record: Dict[str, Any]) -> Tuple[bool, List[str]]:
        errors = []
        vec = record.get("vector")
        if not vec or not isinstance(vec, list):
            return False, ["Vector rỗng hoặc không phải list!"]
        if len(vec) != self.expected_dim:
            errors.append(f"Sai dimension: {len(vec)} != {self.expected_dim}")
        if any(math.isnan(v) or math.isinf(v) for v in vec):
            errors.append("Vector chứa NaN hoặc Inf!")
        norm = math.sqrt(sum(v * v for v in vec))
        if abs(norm - 1.0) > self.l2_tolerance:
            errors.append(f"L2 norm sai lệch: {norm:.6f} (Cho phép: 1.0 ± {self.l2_tolerance})")
        for req_field in ["chunk_id", "source_file", "modality", "model_name"]:
            if not record.get(req_field):
                errors.append(f"Thiếu metadata bắt buộc: '{req_field}'")
        return len(errors) == 0, errors

class CheckpointManager:
    """Quản lý lưu và khôi phục (Resume) batch checkpoint theo mô hình."""
    def __init__(self, checkpoint_dir: Path, dataset_fingerprint: str = "full_chunks"):
        self.checkpoint_dir = checkpoint_dir
        self.dataset_fingerprint = dataset_fingerprint

    def _get_model_dir(self, model_name: str) -> Path:
        mdir = self.checkpoint_dir / model_name
        mdir.mkdir(parents=True, exist_ok=True)
        return mdir

    def load_completed_ids(self, model_name: str, expected_dimension: int) -> Set[str]:
        mdir = self._get_model_dir(model_name)
        completed_ids = set()
        for batch_file in sorted(mdir.glob("batch_*.jsonl")):
            try:
                with open(batch_file, "r", encoding="utf-8") as f:
                    for line in f:
                        if not line.strip():
                            continue
                        rec = json.loads(line)
                        if rec.get("chunk_id") and len(rec.get("vector", [])) == expected_dimension:
                            completed_ids.add(rec["chunk_id"])
            except Exception as e:
                print(f"⚠️ Không thể đọc file batch {batch_file}: {e}")
        return completed_ids

    def save_batch(self, model_name: str, batch_idx: int, records: List[Dict[str, Any]], dimension: int, model_id: str):
        mdir = self._get_model_dir(model_name)
        batch_file = mdir / f"batch_{batch_idx:05d}.jsonl"
        with open(batch_file, "w", encoding="utf-8") as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        manifest_file = mdir / "checkpoint_manifest.json"
        existing_batches = []
        if manifest_file.exists():
            try:
                with open(manifest_file, "r", encoding="utf-8") as mf:
                    existing_batches = json.load(mf).get("batches", [])
            except Exception:
                pass
        existing_batches = [b for b in existing_batches if b.get("batch_idx") != batch_idx]
        existing_batches.append({"batch_idx": batch_idx, "file": batch_file.name, "records": len(records)})
        existing_batches.sort(key=lambda x: x["batch_idx"])
        manifest_data = {
            "model_name": model_name,
            "model_id": model_id,
            "dimension": dimension,
            "dataset_fingerprint": self.dataset_fingerprint,
            "last_updated": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "batches": existing_batches
        }
        with open(manifest_file, "w", encoding="utf-8") as mf:
            json.dump(manifest_data, mf, ensure_ascii=False, indent=2)

    def consolidate(self, model_name: str, output_file: Path) -> int:
        mdir = self._get_model_dir(model_name)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        seen_ids = set()
        total_written = 0
        with open(output_file, "w", encoding="utf-8") as out_f:
            for batch_file in sorted(mdir.glob("batch_*.jsonl")):
                with open(batch_file, "r", encoding="utf-8") as in_f:
                    for line in in_f:
                        if not line.strip():
                            continue
                        rec = json.loads(line)
                        cid = rec.get("chunk_id")
                        if cid and cid not in seen_ids:
                            seen_ids.add(cid)
                            out_f.write(line.strip() + "\n")
                            total_written += 1
        return total_written

print("✅ Đã khởi tạo hoàn tất: ModalityRouter (0.50), VectorValidator, CheckpointManager.")


In [ ]:
# =============================================================================
# 5. NẠP DỮ LIỆU & ĐỊNH TUYẾN TỰ ĐỘNG (DYNAMIC ROUTING)
# =============================================================================
with open(INPUT_CHUNK_FILE, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

assert len(loaded_data) > 0, f"❌ Lỗi: File input '{INPUT_CHUNK_FILE.name}' rỗng!"

print(f"📂 Đã nạp thành công: {len(loaded_data):,} chunks từ {INPUT_CHUNK_FILE.name}")

# Định tuyến tự động theo ngưỡng chuẩn threshold = 0.50
router = ModalityRouter(code_ratio_threshold=0.50)
routed = router.route(loaded_data)
target_text = routed[Modality.TEXT]
target_code = routed[Modality.CODE]

print(f"\n🎯 KẾT QUẢ ĐỊNH TUYẾN DỮ LIỆU ĐẦU VÀO:")
print(f"  • TEXT Chunks (Qwen3)    : {len(target_text):,} chunks ({len(target_text)/len(loaded_data)*100:.2f}%)")
print(f"  • CODE Chunks (Jina Code): {len(target_code):,} chunks ({len(target_code)/len(loaded_data)*100:.2f}%)")
print(f"  • Tổng số Chunks         : {len(target_text) + len(target_code):,} chunks")

# Kiểm tra tính duy nhất của ID
all_input_ids = [c["id"] for c in loaded_data]
assert len(all_input_ids) == len(set(all_input_ids)), "❌ Phát hiện trùng lặp ID trong file input!"
print("🎉 Xác nhận thành công: Toàn bộ Chunk ID duy nhất 100%, sẵn sàng chạy Embedding!")


In [ ]:
# =============================================================================
# 6. TEXT EMBEDDER: QWEN3-EMBEDDING-0.6B (DIM: 1024, LAST-TOKEN POOLING)
# =============================================================================
from transformers import AutoConfig, AutoModel, AutoTokenizer
from tqdm import tqdm

MODEL_ID_QWEN = "Qwen/Qwen3-Embedding-0.6B"
DIM_QWEN = 1024
BATCH_SIZE_QWEN = 32  # Batch size an toàn, tối ưu throughput trên Tesla T4

dataset_fp = f"{INPUT_CHUNK_FILE.stem}_{len(loaded_data)}"
ckpt_mgr = CheckpointManager(CHECKPOINT_DIR, dataset_fingerprint=dataset_fp)
qwen_output_file = EMBEDDINGS_DIR / "qwen3_text" / "qwen3_text_vectors.jsonl"

# BƯỚC 1: XÁC MINH CHECKPOINT ĐÃ CÓ (NẾU ĐÃ CHẠY HOÀN TẤT THÌ BỎ QUA)
qwen_checkpoint_verified = False
qwen_failures = []
total_qwen = 0
qwen_status = "pending"
infer_time_qwen = 0.0
peak_vram_qwen = 0.0

if qwen_output_file.exists():
    print(f"🔍 Đang kiểm tra checkpoint hiện có: {qwen_output_file}...")
    text_ids_set = {c["id"] for c in target_text}
    seen_ids = set()
    val_ok = True
    validator_check = VectorValidator(expected_dim=DIM_QWEN, l2_tolerance=5e-3)
    with open(qwen_output_file, "r", encoding="utf-8") as qf:
        for l in qf:
            if not l.strip(): continue
            rec = json.loads(l)
            cid = rec.get("chunk_id")
            if not cid or cid not in text_ids_set or cid in seen_ids:
                val_ok = False; break
            seen_ids.add(cid)
            is_val, _ = validator_check.validate(rec)
            if not is_val:
                val_ok = False; break
    if val_ok and len(seen_ids) == len(target_text):
        qwen_checkpoint_verified = True
        total_qwen = len(seen_ids)
        qwen_status = "resumed_from_verified_checkpoint"
        print(f"⚡ XÁC MINH THÀNH CÔNG: Đã có đủ {total_qwen:,}/{len(target_text):,} vectors Qwen3 hợp lệ. Bỏ qua nạp model!")

# BƯỚC 2: INFERENCE CÁC CHUNKS CÒN LẠI NẾU CHƯA HOÀN TẤT
if not qwen_checkpoint_verified:
    print(f"🔍 Kiểm tra AutoConfig cho {MODEL_ID_QWEN}...")
    qwen_config = AutoConfig.from_pretrained(MODEL_ID_QWEN)
    assert qwen_config.model_type == "qwen3", f"❌ Lỗi: AutoConfig nhận diện '{qwen_config.model_type}', không phải 'qwen3'!"
    print(f"✅ AutoConfig nhận diện chính xác: model_type='{qwen_config.model_type}'")

    completed_qwen = ckpt_mgr.load_completed_ids("qwen3_text", expected_dimension=DIM_QWEN)
    pending_qwen = [c for c in target_text if c["id"] not in completed_qwen]
    print(f"Qwen3 Text: {len(target_text):,} total -> {len(completed_qwen):,} đã xong -> {len(pending_qwen):,} cần chạy.")

    if pending_qwen:
        start_load_qwen = time.time()
        tokenizer_qwen = AutoTokenizer.from_pretrained(MODEL_ID_QWEN, padding_side="left")
        model_qwen = AutoModel.from_pretrained(MODEL_ID_QWEN, torch_dtype=torch.float16).to("cuda").eval()
        load_time_qwen = time.time() - start_load_qwen
        print(f"✅ Nạp Qwen3 lên CUDA hoàn tất trong {load_time_qwen:.2f}s")

        def qwen3_last_token_pool(last_hidden_states, attention_mask):
            left_padding = bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item())
            if left_padding:
                return last_hidden_states[:, -1]
            else:
                sequence_lengths = attention_mask.sum(dim=1) - 1
                batch_size = last_hidden_states.shape[0]
                return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

        validator_qwen = VectorValidator(expected_dim=DIM_QWEN, l2_tolerance=5e-3)
        start_infer_qwen = time.time()
        torch.cuda.reset_peak_memory_stats()

        num_batches_qwen = (len(pending_qwen) + BATCH_SIZE_QWEN - 1) // BATCH_SIZE_QWEN
        for b_idx in tqdm(range(num_batches_qwen), desc="Embedding Qwen3 Text"):
            batch_chunks = pending_qwen[b_idx * BATCH_SIZE_QWEN : (b_idx + 1) * BATCH_SIZE_QWEN]
            texts = [c.get("text", "") for c in batch_chunks]
            try:
                with torch.no_grad():
                    inputs = tokenizer_qwen(texts, max_length=8192, padding=True, truncation=True, return_tensors="pt").to(model_qwen.device)
                    outputs = model_qwen(**inputs)
                    pooled = qwen3_last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
                    embeddings = torch.nn.functional.normalize(pooled.to(torch.float32), p=2, dim=1)
                    vectors = embeddings.cpu().tolist()
            except torch.cuda.OutOfMemoryError:
                print("⚠️ CUDA OOM! Dọn cache và xử lý từng item nhỏ...")
                torch.cuda.empty_cache()
                vectors = []
                with torch.no_grad():
                    for single_text in texts:
                        s_inp = tokenizer_qwen([single_text], max_length=8192, padding=True, truncation=True, return_tensors="pt").to(model_qwen.device)
                        s_out = model_qwen(**s_inp)
                        s_pooled = qwen3_last_token_pool(s_out.last_hidden_state, s_inp["attention_mask"])
                        s_emb = torch.nn.functional.normalize(s_pooled.to(torch.float32), p=2, dim=1)
                        vectors.append(s_emb.cpu().tolist()[0])

            batch_records = []
            for chunk, vec in zip(batch_chunks, vectors):
                rec = {
                    "chunk_id": chunk["id"],
                    "source_file": chunk.get("metadata", {}).get("source_file", ""),
                    "section_path": chunk.get("metadata", {}).get("section_path", ""),
                    "modality": "text",
                    "model_name": MODEL_ID_QWEN,
                    "model_version": "1.0",
                    "embedding_dimension": len(vec),
                    "vector": vec,
                    "token_count": chunk.get("metadata", {}).get("token_count", 0),
                    "content_preview": chunk.get("text", "")[:120].replace("\n", " "),
                    "metadata": chunk.get("metadata", {})
                }
                is_val, errs = validator_qwen.validate(rec)
                if is_val:
                    batch_records.append(rec)
                else:
                    qwen_failures.append({"id": chunk["id"], "errors": errs})

            if batch_records:
                ckpt_mgr.save_batch("qwen3_text", b_idx, batch_records, DIM_QWEN, MODEL_ID_QWEN)

        infer_time_qwen = time.time() - start_infer_qwen
        peak_vram_qwen = torch.cuda.max_memory_allocated() / (1024 ** 3)
        print("🧹 Giải phóng Qwen3 khỏi GPU VRAM...")
        del model_qwen, tokenizer_qwen
        torch.cuda.empty_cache()
        gc.collect()

    total_qwen = ckpt_mgr.consolidate("qwen3_text", qwen_output_file)
    qwen_status = "inferred_in_current_session"

print(f"\n📊 KẾT QUẢ QWEN3 TEXT:")
print(f"  • Vector hợp lệ : {total_qwen:,}/{len(target_text):,}")
print(f"  • Lỗi QA        : {len(qwen_failures)}")
if infer_time_qwen > 0:
    print(f"  • Thời gian chạy: {infer_time_qwen:.2f}s ({len(target_text)/max(infer_time_qwen, 0.001):.1f} chunks/s)")
    print(f"  • Peak VRAM thực: {peak_vram_qwen:.2f} GB")

assert total_qwen == len(target_text), f"❌ Thiếu vector Qwen3: Đạt {total_qwen:,}/{len(target_text):,}!"


In [ ]:
# =============================================================================
# 7. CODE EMBEDDER: JINA-EMBEDDINGS-V2-BASE-CODE (DIM: 768, ATTENTION-MASKED MEAN)
# =============================================================================
from sentence_transformers import SentenceTransformer

MODEL_ID_JINA = "jinaai/jina-embeddings-v2-base-code"
DIM_JINA = 768
BATCH_SIZE_JINA = 32  # Tối ưu cho GPU Tesla T4 (đo đạc thực tế chỉ tốn ~2.35 GB VRAM)

# COMPATIBILITY MONKEY-PATCH CHO JINABERT TRONG TRANSFORMERS 5.X
try:
    import torch
    import transformers.pytorch_utils
    from transformers.modeling_utils import PreTrainedModel
    from transformers import PretrainedConfig

    if not hasattr(transformers.pytorch_utils, "find_pruneable_heads_and_indices"):
        def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
            mask = torch.ones(n_heads, head_size)
            heads = set(heads) - already_pruned_heads
            for head in heads:
                head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
                mask[head] = 0
            mask = mask.view(-1).contiguous().eq(1)
            index: torch.LongTensor = torch.arange(len(mask))[mask].long()
            return heads, index
        transformers.pytorch_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

    for _attr, _default in [("is_decoder", False), ("add_cross_attention", False), ("chunk_size_feed_forward", 0), ("pruned_heads", set()), ("use_return_dict", True)]:
        if not hasattr(PretrainedConfig, _attr):
            setattr(PretrainedConfig, _attr, _default)

    if not hasattr(PreTrainedModel, "get_head_mask"):
        def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
            if head_mask is not None:
                if head_mask.dim() == 1: head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1).expand(num_hidden_layers, -1, -1, -1, -1)
                elif head_mask.dim() == 2: head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
                return head_mask.to(dtype=getattr(self, "dtype", torch.float32))
            return [None] * num_hidden_layers
        PreTrainedModel.get_head_mask = get_head_mask

    if not hasattr(PreTrainedModel, "get_extended_attention_mask"):
        def get_extended_attention_mask(self, attention_mask, input_shape, dtype=None):
            if dtype is None: dtype = getattr(self, "dtype", torch.float32)
            if attention_mask.dim() == 3: extended_mask = attention_mask[:, None, :, :]
            elif attention_mask.dim() == 2: extended_mask = attention_mask[:, None, None, :]
            else: raise ValueError(f"Wrong attention_mask shape: {attention_mask.shape}")
            extended_mask = extended_mask.to(dtype=dtype)
            return (1.0 - extended_mask) * -10000.0
        PreTrainedModel.get_extended_attention_mask = get_extended_attention_mask

    if not hasattr(PreTrainedModel, "invert_attention_mask"):
        def invert_attention_mask(self, encoder_attention_mask):
            if encoder_attention_mask.dim() == 3: ext = encoder_attention_mask[:, None, :, :]
            elif encoder_attention_mask.dim() == 2: ext = encoder_attention_mask[:, None, None, :]
            else: ext = encoder_attention_mask
            ext = ext.to(dtype=getattr(self, "dtype", torch.float32))
            return (1.0 - ext) * -10000.0
        PreTrainedModel.invert_attention_mask = invert_attention_mask
    print("🔧 Đã kích hoạt Compatibility Patch cho JinaBert trên Transformers 5.x thành công!")
except Exception as e:
    print(f"⚠️ Ghi chú patch: {e}")

jina_output_file = EMBEDDINGS_DIR / "jina_code" / "jina_code_vectors.jsonl"
completed_jina = ckpt_mgr.load_completed_ids("jina_code", expected_dimension=DIM_JINA)
pending_jina = [c for c in target_code if c["id"] not in completed_jina]
print(f"Jina Code: {len(target_code):,} total -> {len(completed_jina):,} đã xong -> {len(pending_jina):,} cần chạy.")

total_jina = len(completed_jina)
infer_time_jina = 0.0
peak_vram_jina = 0.0
jina_failures = []

if pending_jina:
    print(f"=== BẮT ĐẦU NẠP MODEL: {MODEL_ID_JINA} ===")
    start_load_jina = time.time()
    model_jina = SentenceTransformer(MODEL_ID_JINA, trust_remote_code=True, model_kwargs={"attn_implementation": "eager"}, device="cuda")

    # PATCH SAFE EMBEDDINGS FORWARD ĐẢM BẢO TOKEN_TYPE_IDS KHÔNG LỆCH DEVICE VÀ RÁC BUFFER
    def safe_embeddings_forward(self, input_ids=None, token_type_ids=None, position_ids=None, inputs_embeds=None, past_key_values_length=0):
        device = input_ids.device if input_ids is not None else inputs_embeds.device
        input_shape = input_ids.size() if input_ids is not None else inputs_embeds.size()[:-1]
        seq_length = input_shape[1]
        if inputs_embeds is None: inputs_embeds = self.word_embeddings(input_ids)
        safe_tt_ids = torch.zeros(input_shape, dtype=torch.long, device=device)
        try:
            token_type_embeddings = self.token_type_embeddings(safe_tt_ids)
            embeddings = inputs_embeds + token_type_embeddings
        except Exception:
            embeddings = inputs_embeds
        if getattr(self, "position_embedding_type", "alibi") == "absolute":
            if position_ids is None: position_ids = torch.arange(seq_length, dtype=torch.long, device=device).unsqueeze(0).expand(input_shape[0], -1)
            try:
                embeddings += self.position_embeddings(position_ids)
            except Exception:
                pass
        embeddings = self.LayerNorm(embeddings)
        return self.dropout(embeddings)

    try:
        model_jina[0].auto_model.embeddings.__class__.forward = safe_embeddings_forward
        if hasattr(model_jina[0].auto_model.embeddings, "token_type_ids"):
            model_jina[0].auto_model.embeddings.token_type_ids.zero_()
        print("🔧 Đã áp dụng Safe Embeddings Forward patch (triệt tiêu rác buffer token_type_ids).")
    except Exception as e:
        print(f"⚠️ Safe embeddings patch note: {e}")

    load_time_jina = time.time() - start_load_jina
    print(f"✅ Nạp Jina Code lên CUDA hoàn tất trong {load_time_jina:.2f}s")

    # XÁC MINH CẤU HÌNH POOLING CHÍNH THỨC CỦA MODEL
    if len(model_jina) > 1 and hasattr(model_jina[1], "pooling_mode_mean_tokens"):
        print(f"  • Pooling Mode (Attention-Masked Mean Tokens): {model_jina[1].pooling_mode_mean_tokens}")
        assert model_jina[1].pooling_mode_mean_tokens is True, "Cấu hình model không phải Mean Pooling chuẩn!"

    # SMOKE TEST XÁC MINH NHANH
    smoke_samples = ["def binary_search(arr, target):\n    return 0", "class SafeCache:\n    pass"]
    smoke_vectors = model_jina.encode(smoke_samples, batch_size=2, normalize_embeddings=True, show_progress_bar=False).tolist()
    assert len(smoke_vectors) == 2 and len(smoke_vectors[0]) == 768, "❌ Smoke test thất bại!"
    print("🎉 Smoke test hoàn toàn thành công trên CUDA!")

    validator_jina = VectorValidator(expected_dim=DIM_JINA, l2_tolerance=5e-3)
    start_infer_jina = time.time()
    torch.cuda.reset_peak_memory_stats()

    num_batches_jina = (len(pending_jina) + BATCH_SIZE_JINA - 1) // BATCH_SIZE_JINA
    for b_idx in tqdm(range(num_batches_jina), desc="Embedding Jina Code"):
        batch_chunks = pending_jina[b_idx * BATCH_SIZE_JINA : (b_idx + 1) * BATCH_SIZE_JINA]
        texts = [c.get("text", "") for c in batch_chunks]
        try:
            embeddings = model_jina.encode(texts, batch_size=BATCH_SIZE_JINA, normalize_embeddings=True, show_progress_bar=False).tolist()
        except Exception as e:
            print(f"⚠️ Lỗi batch Jina ({e}), fallback sang batch nhỏ 8...")
            torch.cuda.empty_cache()
            embeddings = model_jina.encode(texts, batch_size=8, normalize_embeddings=True, show_progress_bar=False).tolist()

        batch_records = []
        for chunk, vec in zip(batch_chunks, embeddings):
            raw_text = chunk.get("text", "")
            rec = {
                "chunk_id": chunk["id"],
                "source_file": chunk.get("metadata", {}).get("source_file", ""),
                "section_path": chunk.get("metadata", {}).get("section_path", ""),
                "modality": "code",
                "model_name": MODEL_ID_JINA,
                "model_version": "2.0",
                "embedding_dimension": len(vec),
                "vector": vec,
                "token_count": chunk.get("metadata", {}).get("token_count", 0),
                "content_preview": raw_text[:120].replace("\n", " "),
                "metadata": chunk.get("metadata", {})
            }
            is_val, errs = validator_jina.validate(rec)
            if is_val:
                batch_records.append(rec)
            else:
                jina_failures.append({"id": chunk["id"], "errors": errs})

        if batch_records:
            ckpt_mgr.save_batch("jina_code", b_idx, batch_records, DIM_JINA, MODEL_ID_JINA)

    infer_time_jina = time.time() - start_infer_jina
    peak_vram_jina = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print("🧹 Giải phóng Jina Code khỏi GPU VRAM...")
    del model_jina
    torch.cuda.empty_cache()
    gc.collect()

total_jina = ckpt_mgr.consolidate("jina_code", jina_output_file)

print(f"\n📊 KẾT QUẢ JINA CODE:")
print(f"  • Vector hợp lệ : {total_jina:,}/{len(target_code):,}")
print(f"  • Lỗi QA        : {len(jina_failures)}")
if infer_time_jina > 0:
    print(f"  • Thời gian chạy: {infer_time_jina:.2f}s ({len(target_code)/max(infer_time_jina, 0.001):.1f} chunks/s)")
    print(f"  • Peak VRAM thực: {peak_vram_jina:.2f} GB")

assert total_jina == len(target_code), f"❌ Thiếu vector Jina Code: Đạt {total_jina:,}/{len(target_code):,}!"


In [ ]:
# =============================================================================
# 8. TỔNG KẾT PIPELINE, XUẤT BÁO CÁO JSON & ĐÓNG GÓI BẢN PHÁT HÀNH
# =============================================================================
import tarfile

qwen_ids = set()
if qwen_output_file.exists():
    with open(qwen_output_file, "r", encoding="utf-8") as f:
        for l in f:
            if l.strip():
                qwen_ids.add(json.loads(l)["chunk_id"])

jina_ids = set()
if jina_output_file.exists():
    with open(jina_output_file, "r", encoding="utf-8") as f:
        for l in f:
            if l.strip():
                jina_ids.add(json.loads(l)["chunk_id"])

print(f"\n🔍 KIỂM TOÁN TỔNG THỂ DỮ LIỆU ĐẦU RA:")
print(f"  • Qwen3 Text Vectors : {len(qwen_ids):,} (Kỳ vọng: {len(target_text):,})")
print(f"  • Jina Code Vectors  : {len(jina_ids):,} (Kỳ vọng: {len(target_code):,})")
print(f"  • Tổng số Vectors    : {len(qwen_ids) + len(jina_ids):,} (Kỳ vọng: {len(loaded_data):,})")

assert len(qwen_ids) == len(target_text), f"❌ Thiếu vector Qwen3: {len(qwen_ids):,} != {len(target_text):,}!"
assert len(jina_ids) == len(target_code), f"❌ Thiếu vector Jina: {len(jina_ids):,} != {len(target_code):,}!"

overlap_ids = qwen_ids.intersection(jina_ids)
assert len(overlap_ids) == 0, f"❌ Phát hiện trùng lặp ID giữa hai model: {overlap_ids}"
print("✅ Hai tập ID hoàn toàn tách biệt (0 overlap), khớp 100% với tập chunks đầu vào!")

report_data = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "input_file": INPUT_CHUNK_FILE.name,
    "total_chunks_input": len(loaded_data),
    "qwen3_text": {
        "model_id": MODEL_ID_QWEN,
        "target_chunks": len(target_text),
        "completed_vectors": total_qwen,
        "dimension": DIM_QWEN,
        "pooling_mode": "last_token_pooling",
        "padding_side": "left",
        "status": qwen_status,
        "failures_count": len(qwen_failures),
        "inference_time_sec": round(infer_time_qwen, 2),
        "peak_vram_gb": round(peak_vram_qwen, 2)
    },
    "jina_code": {
        "model_id": MODEL_ID_JINA,
        "target_chunks": len(target_code),
        "completed_vectors": total_jina,
        "dimension": DIM_JINA,
        "pooling_mode": "attention_masked_mean_tokens",
        "pooling_description": "Attention-Masked Mean Pooling (không sử dụng token weighting riêng)",
        "status": "completed",
        "failures_count": len(jina_failures),
        "inference_time_sec": round(infer_time_jina, 2),
        "peak_vram_gb": round(peak_vram_jina, 2)
    },
    "overall": {
        "total_embedded": total_qwen + total_jina,
        "success_rate": round((total_qwen + total_jina) / max(len(loaded_data), 1) * 100, 2),
        "id_collision_count": len(overlap_ids),
        "status": "SUCCESS"
    }
}

report_path = REPORTS_DIR / "kaggle_embedding_benchmark_report.json"
with open(report_path, "w", encoding="utf-8") as rf:
    json.dump(report_data, rf, ensure_ascii=False, indent=2)

print("\n📊 BÁO CÁO NGHIỆM THU BENCHMARK:")
print(json.dumps(report_data, ensure_ascii=False, indent=2))

# Đóng gói nén tar.gz tại /kaggle/working/
archive_path = WORKING_DIR / "ai_coding_tutor_embeddings_bundle.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(EMBEDDINGS_DIR, arcname="embeddings")
    tar.add(REPORTS_DIR, arcname="reports")
    tar.add(CHECKPOINT_DIR, arcname="checkpoints")

print(f"\n📦 ĐÃ ĐÓNG GÓI TOÀN BỘ KẾT QUẢ THÀNH CÔNG: {archive_path} ({archive_path.stat().st_size:,} bytes)")
print("👉 Bạn có thể tải file này trực tiếp từ tab 'Output' của Kaggle Notebook về máy tính!")
